# Classical ML Models

## Goal

Evaluate classical machine learning models for privacy sensitive prompt detection.

Labels:
- 0 = safe (no PII)
- 1 = privacy sensitive (contains PII)

This notebook trains and evaluates the following models:
- Logistic Regression
- Naive Bayes
- Linear SVM



In [32]:
# Imports
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import MultinomialNB, BernoulliNB, ComplementNB
from sklearn.svm import LinearSVC 
from pathlib import Path
import numpy as np
from scipy.sparse import load_npz
from sklearn.base import clone
from sklearn.model_selection import ParameterGrid
import time
import os
import json

## Load Frozen Data Splits
Train, validation, and test sets generated by 03_feature_engineering.ipynb.

In [33]:
FEATURE_DIR = Path('../../feature_matrices')
train_features = load_npz(FEATURE_DIR / 'X_train_combined.npz')
val_features = load_npz(FEATURE_DIR / 'X_val_combined.npz')
test_features = load_npz(FEATURE_DIR / 'X_test_combined.npz')

train_labels = np.load(FEATURE_DIR / 'y_train.npy')
val_labels = np.load(FEATURE_DIR / 'y_val.npy')
test_labels = np.load(FEATURE_DIR / 'y_test.npy')

In [34]:
# data set verification sanity check
print(train_features.shape)
print(val_features.shape)
print(test_features.shape)

(260338, 50018)
(32542, 50018)
(32543, 50018)


## Baseline Model: Always Predict PII

As a simple baseline, we evaluate a model that predicts every prompt as PII. Since the dataset has more sensitive examples than safe examples, this baseline helps show whether the trained models are learning beyond the class imbalance.

In [35]:
# create array of 1s to serve as the baseline
baseline_predictions = np.ones(len(val_labels), dtype=int)
# zero_division set to 0 to hide warning due to zero examples being predicted as safe
print(classification_report(val_labels, baseline_predictions, zero_division=0))

cm = confusion_matrix(val_labels, baseline_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

           0       0.00      0.00      0.00     10616
           1       0.67      1.00      0.81     21926

    accuracy                           0.67     32542
   macro avg       0.34      0.50      0.40     32542
weighted avg       0.45      0.67      0.54     32542



,Predicted Safe,Predicted PII
Actual Safe,0,10616
Actual PII,0,21926


## Logistic Regression

Logistic Regression is a common baseline for text classification because it performs well on sparse TF-IDF representations and produces interpretable feature weights. However, the model learns a linear decision boundary and may struggle to capture complex relationships between terms.

https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

https://www.geeksforgeeks.org/machine-learning/understanding-logistic-regression/

The model computes a weighted sum of the input features:

$$
z = \sum_{i=1}^{n} w_i x_i + b
$$

The sigmoid function maps this score to the range [0, 1]:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

The resulting value is interpreted as the probability that a prompt belongs to the privacy sensitive class:

$$
P(y=1|x) = \sigma(z)
$$

### Tuning

In [36]:
# leaving outside for if combined comparison is made later
results = []
save_path = 'classical_tuning_results.csv'

# load data from csv
if os.path.exists(save_path):
    results_df = pd.read_csv(save_path)
    results = results_df.to_dict('records')
    completed_keys = set(results_df['key'])
    print(f'Loaded {len(results)} saved results from CSV.')
else:
    completed_keys = set()
    print('No saved results found.')

# helper function to make keys
def make_key(model_name, params):
    return f"{model_name}|{json.dumps(params, sort_keys=True)}"

# helper function for tuning the classical models

def run_search(model_name, base_model, param_grid):
    configs = list(ParameterGrid(param_grid))

    print(f'\n----- {model_name} tuning started -----')
    print(f'Total configs for {model_name}: {len(configs)}')

    for i, params in enumerate(configs, start=1):
        # if already in csv, skip to save time
        key = make_key(model_name, params)

        if key in completed_keys:
            print(f'Skipping {model_name} {i}/{len(configs)}: already saved')
            continue

        print('\n' + '-' * 60)
        print(f'{model_name} progress: {i}/{len(configs)}')
        print(f'Params: {params}')

        # clone the base model and use this iteration's params
        model = clone(base_model)
        model.set_params(**params)
        
        # keep track of time to find bottlenecks
        start = time.perf_counter()

        # train model
        model.fit(train_features, train_labels)
        preds = model.predict(val_features)

        elapsed = (time.perf_counter() - start) / 60

        row = {
            'model': model_name,
            'params': str(params),
            'key': key,
            **params,
            'accuracy': accuracy_score(val_labels, preds),
            'precision': precision_score(val_labels, preds),
            'recall': recall_score(val_labels, preds),
            'f1': f1_score(val_labels, preds),
            'time_min': elapsed
        }

        results.append(row)
        completed_keys.add(key)
        # save to csv so that this doesn't need to be run everytime
        results_df = pd.DataFrame(results)
        results_df = results_df.drop_duplicates(subset=['key'], keep='last')
        results_df.to_csv(save_path, index=False)

        results_df = pd.DataFrame(results).sort_values('f1', ascending=False)
        # best = results_df.iloc[0]

        # progress tracking (made the mistake of not doing this last time)
        print(
            f"Finished in {elapsed:.2f} min\n"
            f"F1:        {row['f1']:.4f}\n"
            f"Precision: {row['precision']:.4f}\n"
            f"Recall:    {row['recall']:.4f}"
        )

        # print(f"Best overall so far: {best['model']} | F1={best['f1']:.4f}")

    print(f'\n----- {model_name} tuning complete -----')

    model_results = (pd.DataFrame(results).query('model == @model_name').sort_values('f1', ascending=False))

    print(f'\nTop results for {model_name}:')
    display(model_results.head(10))

Loaded 112 saved results from CSV.


In [37]:
# saga with elastic was taking too long so the solver has been switched over to liblinear
run_search(
    model_name='Logistic Regression',
    base_model=LogisticRegression(solver='liblinear', max_iter=1000, random_state=42),
    param_grid={'C': [0.1, 1, 2.5, 2.6, 2.65, 2.7, 2.75, 2.8, 2.85, 2.9, 2.95, 3, 3.25, 3.5, 5, 10], 'l1_ratio': [0.0, 1.0]}
)


----- Logistic Regression tuning started -----
Total configs for Logistic Regression: 32
Skipping Logistic Regression 1/32: already saved
Skipping Logistic Regression 2/32: already saved
Skipping Logistic Regression 3/32: already saved
Skipping Logistic Regression 4/32: already saved
Skipping Logistic Regression 5/32: already saved
Skipping Logistic Regression 6/32: already saved
Skipping Logistic Regression 7/32: already saved
Skipping Logistic Regression 8/32: already saved
Skipping Logistic Regression 9/32: already saved
Skipping Logistic Regression 10/32: already saved
Skipping Logistic Regression 11/32: already saved
Skipping Logistic Regression 12/32: already saved
Skipping Logistic Regression 13/32: already saved
Skipping Logistic Regression 14/32: already saved
Skipping Logistic Regression 15/32: already saved
Skipping Logistic Regression 16/32: already saved
Skipping Logistic Regression 17/32: already saved
Skipping Logistic Regression 18/32: already saved
Skipping Logistic R

,model,params,key,C,l1_ratio,accuracy,precision,recall,f1,time_min,alpha,fit_prior,norm,loss
9,Logistic Regression,"{'C': 2.65, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.65, ""l1_ratio"": 1.0}",2.65,1.0,0.830342,0.869133,0.880826,0.874941,0.917613,NaN,NaN,NaN,NaN
11,Logistic Regression,"{'C': 2.7, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.7, ""l1_ratio"": 1.0}",2.70,1.0,0.830281,0.869021,0.880872,0.874907,0.894487,NaN,NaN,NaN,NaN
13,Logistic Regression,"{'C': 2.75, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.75, ""l1_ratio"": 1.0}",2.75,1.0,0.830004,0.868869,0.880598,0.874694,0.942667,NaN,NaN,NaN,NaN
5,Logistic Regression,"{'C': 2.5, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.5, ""l1_ratio"": 1.0}",2.50,1.0,0.829974,0.868763,0.880690,0.874686,0.827407,NaN,NaN,NaN,NaN
7,Logistic Regression,"{'C': 2.6, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.6, ""l1_ratio"": 1.0}",2.60,1.0,0.829943,0.868857,0.880507,0.874643,0.831703,NaN,NaN,NaN,NaN
15,Logistic Regression,"{'C': 2.8, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.8, ""l1_ratio"": 1.0}",2.80,1.0,0.829758,0.868821,0.880234,0.874490,0.949654,NaN,NaN,NaN,NaN
19,Logistic Regression,"{'C': 2.9, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.9, ""l1_ratio"": 1.0}",2.90,1.0,0.829666,0.868505,0.880507,0.874465,0.859714,NaN,NaN,NaN,NaN
17,Logistic Regression,"{'C': 2.85, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.85, ""l1_ratio"": 1.0}",2.85,1.0,0.829697,0.868677,0.880325,0.874462,0.970241,NaN,NaN,NaN,NaN
21,Logistic Regression,"{'C': 2.95, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 2.95, ""l1_ratio"": 1.0}",2.95,1.0,0.829205,0.868019,0.880370,0.874151,0.870805,NaN,NaN,NaN,NaN
23,Logistic Regression,"{'C': 3, 'l1_ratio': 1.0}","Logistic Regression|{""C"": 3, ""l1_ratio"": 1.0}",3.00,1.0,0.829113,0.868166,0.880005,0.874046,0.888502,NaN,NaN,NaN,NaN


### Training

Fit the logistic regression model using the engineered combined feature representation of the training data.
Use set seed (42) to ensure reproducible results across runs / team members. 

In [38]:
# reused seed used in data split script
lr = LogisticRegression(C=2.65, l1_ratio=1, random_state=42, max_iter=1000, solver='liblinear')
lr.fit(train_features, train_labels)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",2.65
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;-

### Evaluation

Generate predictions on the validation set for model evaluation, then evaluate model performance using a classification report and confusion matrix.

For this project, special attention should go towards false negatives since they represent privacy sensitive prompts incorrectly classified as safe.

[![Confusion Matrix](https://i0.wp.com/statisticsbyjim.com/wp-content/uploads/2025/05/confusion_matrix-1.png?fit=550%2C450&ssl=1)](https://statisticsbyjim.com/glossary/confusion-matrix/)

In [39]:
lr_predictions = lr.predict(val_features)
print(classification_report(val_labels, lr_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(val_labels, lr_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.75      0.73      0.74     10616
         PII       0.87      0.88      0.87     21926

    accuracy                           0.83     32542
   macro avg       0.81      0.80      0.81     32542
weighted avg       0.83      0.83      0.83     32542



,Predicted Safe,Predicted PII
Actual Safe,7708,2908
Actual PII,2613,19313


## Naive Bayes

Multinomial Naive Bayes is a probabilistic classification algorithm also commonly used for text classification.

The model estimates the probability that a prompt belongs to each class based on the observed features and predicts the class with the highest posterior probability.

https://scikit-learn.org/stable/modules/naive_bayes.html

https://scikit-learn.org/stable/api/sklearn.naive_bayes.html

Bayes' Theorem:

$$
P(y|x)=\frac{P(x|y)P(y)}{P(x)}
$$

Naive Bayes assumes that features are conditionally independent given the class label.

$$
P(x_1,x_2,\ldots,x_n|y)
=
\prod_{i=1}^{n} P(x_i|y)
$$

In other words, once the model knows whether a prompt belongs to the Safe or PII class, it treats each feature as contributing independently to the final prediction. While this assumption is rarely true for real-world text data, it greatly simplifies computation and often performs surprisingly well for text classification tasks.

### Tuning

In [40]:
run_search(
    model_name='MultinomialNB',
    base_model=MultinomialNB(),
    param_grid={'alpha': [0.0001, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0]}
)

run_search(
    model_name='BernoulliNB',
    base_model=BernoulliNB(binarize=0.0),
    param_grid={'alpha': [0.0001, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0]}
)

run_search(
    model_name='ComplementNB',
    base_model=ComplementNB(),
    param_grid={'alpha': [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1], 'fit_prior': [True, False], 'norm': [False, True]}
)


----- MultinomialNB tuning started -----
Total configs for MultinomialNB: 7
Skipping MultinomialNB 1/7: already saved
Skipping MultinomialNB 2/7: already saved
Skipping MultinomialNB 3/7: already saved
Skipping MultinomialNB 4/7: already saved
Skipping MultinomialNB 5/7: already saved
Skipping MultinomialNB 6/7: already saved
Skipping MultinomialNB 7/7: already saved

----- MultinomialNB tuning complete -----

Top results for MultinomialNB:


,model,params,key,C,l1_ratio,accuracy,precision,recall,f1,time_min,alpha,fit_prior,norm,loss
32,MultinomialNB,{'alpha': 0.0001},"MultinomialNB|{""alpha"": 0.0001}",NaN,NaN,0.750630,0.808212,0.825869,0.816945,0.005240,0.0001,NaN,NaN,NaN
33,MultinomialNB,{'alpha': 0.001},"MultinomialNB|{""alpha"": 0.001}",NaN,NaN,0.750538,0.808214,0.825686,0.816857,0.001567,0.0010,NaN,NaN,NaN
34,MultinomialNB,{'alpha': 0.01},"MultinomialNB|{""alpha"": 0.01}",NaN,NaN,0.750169,0.808194,0.825002,0.816512,0.001550,0.0100,NaN,NaN,NaN
35,MultinomialNB,{'alpha': 0.05},"MultinomialNB|{""alpha"": 0.05}",NaN,NaN,0.749554,0.808050,0.824045,0.815969,0.001561,0.0500,NaN,NaN,NaN
36,MultinomialNB,{'alpha': 0.1},"MultinomialNB|{""alpha"": 0.1}",NaN,NaN,0.749063,0.807747,0.823588,0.815591,0.001537,0.1000,NaN,NaN,NaN
38,MultinomialNB,{'alpha': 1.0},"MultinomialNB|{""alpha"": 1.0}",NaN,NaN,0.744822,0.798388,0.831159,0.814444,0.001680,1.0000,NaN,NaN,NaN
37,MultinomialNB,{'alpha': 0.5},"MultinomialNB|{""alpha"": 0.5}",NaN,NaN,0.746174,0.803878,0.824409,0.814014,0.001752,0.5000,NaN,NaN,NaN



----- BernoulliNB tuning started -----
Total configs for BernoulliNB: 7
Skipping BernoulliNB 1/7: already saved
Skipping BernoulliNB 2/7: already saved
Skipping BernoulliNB 3/7: already saved
Skipping BernoulliNB 4/7: already saved
Skipping BernoulliNB 5/7: already saved
Skipping BernoulliNB 6/7: already saved
Skipping BernoulliNB 7/7: already saved

----- BernoulliNB tuning complete -----

Top results for BernoulliNB:


,model,params,key,C,l1_ratio,accuracy,precision,recall,f1,time_min,alpha,fit_prior,norm,loss
39,BernoulliNB,{'alpha': 0.0001},"BernoulliNB|{""alpha"": 0.0001}",NaN,NaN,0.736187,0.857036,0.730275,0.788594,0.004720,0.0001,NaN,NaN,NaN
40,BernoulliNB,{'alpha': 0.001},"BernoulliNB|{""alpha"": 0.001}",NaN,NaN,0.736033,0.857074,0.729955,0.788424,0.003256,0.0010,NaN,NaN,NaN
41,BernoulliNB,{'alpha': 0.01},"BernoulliNB|{""alpha"": 0.01}",NaN,NaN,0.735327,0.857013,0.728769,0.787706,0.003268,0.0100,NaN,NaN,NaN
42,BernoulliNB,{'alpha': 0.05},"BernoulliNB|{""alpha"": 0.05}",NaN,NaN,0.734313,0.857028,0.726945,0.786645,0.003146,0.0500,NaN,NaN,NaN
43,BernoulliNB,{'alpha': 0.1},"BernoulliNB|{""alpha"": 0.1}",NaN,NaN,0.733114,0.856843,0.725030,0.785444,0.003469,0.1000,NaN,NaN,NaN
44,BernoulliNB,{'alpha': 0.5},"BernoulliNB|{""alpha"": 0.5}",NaN,NaN,0.731025,0.856856,0.721290,0.783250,0.003363,0.5000,NaN,NaN,NaN
45,BernoulliNB,{'alpha': 1.0},"BernoulliNB|{""alpha"": 1.0}",NaN,NaN,0.729642,0.855965,0.719876,0.782044,0.003203,1.0000,NaN,NaN,NaN



----- ComplementNB tuning started -----
Total configs for ComplementNB: 24
Skipping ComplementNB 1/24: already saved
Skipping ComplementNB 2/24: already saved
Skipping ComplementNB 3/24: already saved
Skipping ComplementNB 4/24: already saved
Skipping ComplementNB 5/24: already saved
Skipping ComplementNB 6/24: already saved
Skipping ComplementNB 7/24: already saved
Skipping ComplementNB 8/24: already saved
Skipping ComplementNB 9/24: already saved
Skipping ComplementNB 10/24: already saved
Skipping ComplementNB 11/24: already saved
Skipping ComplementNB 12/24: already saved
Skipping ComplementNB 13/24: already saved
Skipping ComplementNB 14/24: already saved
Skipping ComplementNB 15/24: already saved
Skipping ComplementNB 16/24: already saved
Skipping ComplementNB 17/24: already saved
Skipping ComplementNB 18/24: already saved
Skipping ComplementNB 19/24: already saved
Skipping ComplementNB 20/24: already saved
Skipping ComplementNB 21/24: already saved
Skipping ComplementNB 22/24: a

,model,params,key,C,l1_ratio,accuracy,precision,recall,f1,time_min,alpha,fit_prior,norm,loss
46,ComplementNB,"{'alpha': 0.0001, 'fit_prior': True, 'norm': F...","ComplementNB|{""alpha"": 0.0001, ""fit_prior"": tr...",NaN,NaN,0.734067,0.878767,0.702180,0.780611,0.002659,0.0001,True,False,NaN
48,ComplementNB,"{'alpha': 0.0001, 'fit_prior': False, 'norm': ...","ComplementNB|{""alpha"": 0.0001, ""fit_prior"": fa...",NaN,NaN,0.734067,0.878767,0.702180,0.780611,0.001517,0.0001,False,False,NaN
52,ComplementNB,"{'alpha': 0.001, 'fit_prior': False, 'norm': F...","ComplementNB|{""alpha"": 0.001, ""fit_prior"": fal...",NaN,NaN,0.733637,0.878887,0.701314,0.780123,0.001387,0.0010,False,False,NaN
50,ComplementNB,"{'alpha': 0.001, 'fit_prior': True, 'norm': Fa...","ComplementNB|{""alpha"": 0.001, ""fit_prior"": tru...",NaN,NaN,0.733637,0.878887,0.701314,0.780123,0.001364,0.0010,True,False,NaN
56,ComplementNB,"{'alpha': 0.005, 'fit_prior': False, 'norm': F...","ComplementNB|{""alpha"": 0.005, ""fit_prior"": fal...",NaN,NaN,0.732838,0.878880,0.699945,0.779273,0.001363,0.0050,False,False,NaN
54,ComplementNB,"{'alpha': 0.005, 'fit_prior': True, 'norm': Fa...","ComplementNB|{""alpha"": 0.005, ""fit_prior"": tru...",NaN,NaN,0.732838,0.878880,0.699945,0.779273,0.001464,0.0050,True,False,NaN
58,ComplementNB,"{'alpha': 0.01, 'fit_prior': True, 'norm': False}","ComplementNB|{""alpha"": 0.01, ""fit_prior"": true...",NaN,NaN,0.732377,0.879036,0.698988,0.778740,0.001376,0.0100,True,False,NaN
60,ComplementNB,"{'alpha': 0.01, 'fit_prior': False, 'norm': Fa...","ComplementNB|{""alpha"": 0.01, ""fit_prior"": fals...",NaN,NaN,0.732377,0.879036,0.698988,0.778740,0.001353,0.0100,False,False,NaN
67,ComplementNB,"{'alpha': 0.1, 'fit_prior': True, 'norm': True}","ComplementNB|{""alpha"": 0.1, ""fit_prior"": true,...",NaN,NaN,0.730809,0.875613,0.699900,0.777958,0.001450,0.1000,True,True,NaN
69,ComplementNB,"{'alpha': 0.1, 'fit_prior': False, 'norm': True}","ComplementNB|{""alpha"": 0.1, ""fit_prior"": false...",NaN,NaN,0.730809,0.875613,0.699900,0.777958,0.001589,0.1000,False,True,NaN


### Training

Fit the Naive Bayes model using the engineered combined feature representation of the training data.

In [41]:
nb = MultinomialNB(alpha=0.0001)
nb.fit(train_features, train_labels)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",0.0001
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[ 84930.,175408.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-1.12,-0.39]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 50018)","[[ 383.17, 151.52, 1.81,..., 2921. , 529. , 528. ], [ 1306.95, 596.55, 5.14,...,12122. , 6996. , 3669. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 50018)","[[ -7.19, -8.12,-12.55,..., -5.16, -6.87, -6.87], [ -6.87, -7.65,-12.4 ,..., -4.64, -5.19, -5.83]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,50018


### Evaluation

Generate predictions on the validation set for model evaluation, then evaluate model performance using a classification report and confusion matrix.

In [42]:
nb_predictions = nb.predict(val_features)
print(classification_report(val_labels, nb_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(val_labels, nb_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.62      0.60      0.61     10616
         PII       0.81      0.83      0.82     21926

    accuracy                           0.75     32542
   macro avg       0.72      0.71      0.71     32542
weighted avg       0.75      0.75      0.75     32542



,Predicted Safe,Predicted PII
Actual Safe,6319,4297
Actual PII,3818,18108


## Linear Support Vector Machine (SVM)

A Linear Support Vector Machine is a classification model that tries to find a decision boundary separating the Safe and PII classes with the largest possible margin.

https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html

https://www.geeksforgeeks.org/machine-learning/support-vector-machine-algorithm/

For a linear decision boundary, the model computes:

$$
f(x) = w^T x + b
$$

- Class 0: f(x) > 0
- Class 1: f(x) < 0

Predictions are based on which side of the decision boundary the example falls on.


### Tuning

In [43]:
run_search(
    model_name='LinearSVM',
    base_model=LinearSVC(dual='auto', max_iter=5000, random_state=42),
    param_grid={'C': [0.01, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 1, 1.1, 1.2, 1.25, 1.4, 1.5, 1.6, 1.8, 2]}
)

run_search(
    model_name='LinearSVM Loss Check',
    base_model=LinearSVC(dual='auto', max_iter=5000, random_state=42),
    param_grid={'C': [0.01, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 1, 1.1, 1.2, 1.25, 1.5, 2], 'loss': ['hinge', 'squared_hinge']}
)


----- LinearSVM tuning started -----
Total configs for LinearSVM: 16
Skipping LinearSVM 1/16: already saved
Skipping LinearSVM 2/16: already saved
Skipping LinearSVM 3/16: already saved
Skipping LinearSVM 4/16: already saved
Skipping LinearSVM 5/16: already saved
Skipping LinearSVM 6/16: already saved
Skipping LinearSVM 7/16: already saved
Skipping LinearSVM 8/16: already saved
Skipping LinearSVM 9/16: already saved
Skipping LinearSVM 10/16: already saved
Skipping LinearSVM 11/16: already saved
Skipping LinearSVM 12/16: already saved
Skipping LinearSVM 13/16: already saved
Skipping LinearSVM 14/16: already saved
Skipping LinearSVM 15/16: already saved
Skipping LinearSVM 16/16: already saved

----- LinearSVM tuning complete -----

Top results for LinearSVM:


,model,params,key,C,l1_ratio,accuracy,precision,recall,f1,time_min,alpha,fit_prior,norm,loss
102,LinearSVM,{'C': 0.25},"LinearSVM|{""C"": 0.25}",0.25,NaN,0.826931,0.862928,0.883472,0.873079,1.002402,NaN,NaN,NaN,NaN
70,LinearSVM,{'C': 0.5},"LinearSVM|{""C"": 0.5}",0.50,NaN,0.827362,0.865880,0.880097,0.872930,0.798672,NaN,NaN,NaN,NaN
71,LinearSVM,{'C': 0.75},"LinearSVM|{""C"": 0.75}",0.75,NaN,0.827024,0.866670,0.878409,0.872500,0.848351,NaN,NaN,NaN,NaN
73,LinearSVM,{'C': 0.9},"LinearSVM|{""C"": 0.9}",0.90,NaN,0.826839,0.867063,0.877543,0.872271,1.280241,NaN,NaN,NaN,NaN
72,LinearSVM,{'C': 0.8},"LinearSVM|{""C"": 0.8}",0.80,NaN,0.826747,0.866781,0.877771,0.872241,1.385538,NaN,NaN,NaN,NaN
74,LinearSVM,{'C': 1},"LinearSVM|{""C"": 1}",1.00,NaN,0.826009,0.866736,0.876539,0.871610,1.359025,NaN,NaN,NaN,NaN
75,LinearSVM,{'C': 1.1},"LinearSVM|{""C"": 1.1}",1.10,NaN,0.825610,0.866426,0.876266,0.871318,1.075839,NaN,NaN,NaN,NaN
76,LinearSVM,{'C': 1.2},"LinearSVM|{""C"": 1.2}",1.20,NaN,0.825057,0.866450,0.875262,0.870834,1.318827,NaN,NaN,NaN,NaN
77,LinearSVM,{'C': 1.25},"LinearSVM|{""C"": 1.25}",1.25,NaN,0.825026,0.866411,0.875262,0.870814,1.373226,NaN,NaN,NaN,NaN
101,LinearSVM,{'C': 0.1},"LinearSVM|{""C"": 0.1}",0.10,NaN,0.822629,0.855064,0.887120,0.870797,0.613948,NaN,NaN,NaN,NaN



----- LinearSVM Loss Check tuning started -----
Total configs for LinearSVM Loss Check: 26
Skipping LinearSVM Loss Check 1/26: already saved
Skipping LinearSVM Loss Check 2/26: already saved
Skipping LinearSVM Loss Check 3/26: already saved
Skipping LinearSVM Loss Check 4/26: already saved
Skipping LinearSVM Loss Check 5/26: already saved
Skipping LinearSVM Loss Check 6/26: already saved
Skipping LinearSVM Loss Check 7/26: already saved
Skipping LinearSVM Loss Check 8/26: already saved
Skipping LinearSVM Loss Check 9/26: already saved
Skipping LinearSVM Loss Check 10/26: already saved
Skipping LinearSVM Loss Check 11/26: already saved
Skipping LinearSVM Loss Check 12/26: already saved
Skipping LinearSVM Loss Check 13/26: already saved
Skipping LinearSVM Loss Check 14/26: already saved
Skipping LinearSVM Loss Check 15/26: already saved
Skipping LinearSVM Loss Check 16/26: already saved
Skipping LinearSVM Loss Check 17/26: already saved
Skipping LinearSVM Loss Check 18/26: already saved

,model,params,key,C,l1_ratio,accuracy,precision,recall,f1,time_min,alpha,fit_prior,norm,loss
96,LinearSVM Loss Check,"{'C': 1.5, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 1.5, ""loss"": ""hinge""}",1.50,NaN,0.830496,0.868367,0.882149,0.875204,1.062098,NaN,NaN,NaN,hinge
86,LinearSVM Loss Check,"{'C': 0.9, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 0.9, ""loss"": ""hinge""}",0.90,NaN,0.829697,0.866664,0.883107,0.874808,0.594791,NaN,NaN,NaN,hinge
88,LinearSVM Loss Check,"{'C': 1, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 1, ""loss"": ""hinge""}",1.00,NaN,0.829267,0.866482,0.882605,0.874469,1.089667,NaN,NaN,NaN,hinge
82,LinearSVM Loss Check,"{'C': 0.75, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 0.75, ""loss"": ""hinge""}",0.75,NaN,0.828929,0.865270,0.883700,0.874388,0.292611,NaN,NaN,NaN,hinge
84,LinearSVM Loss Check,"{'C': 0.8, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 0.8, ""loss"": ""hinge""}",0.80,NaN,0.828990,0.865642,0.883289,0.874376,0.545236,NaN,NaN,NaN,hinge
92,LinearSVM Loss Check,"{'C': 1.2, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 1.2, ""loss"": ""hinge""}",1.20,NaN,0.829144,0.867148,0.881465,0.874248,0.484706,NaN,NaN,NaN,hinge
94,LinearSVM Loss Check,"{'C': 1.25, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 1.25, ""loss"": ""hinge""}",1.25,NaN,0.829021,0.866697,0.881875,0.874220,0.470282,NaN,NaN,NaN,hinge
90,LinearSVM Loss Check,"{'C': 1.1, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 1.1, ""loss"": ""hinge""}",1.10,NaN,0.828990,0.866592,0.881967,0.874212,1.150829,NaN,NaN,NaN,hinge
98,LinearSVM Loss Check,"{'C': 2, 'loss': 'hinge'}","LinearSVM Loss Check|{""C"": 2, ""loss"": ""hinge""}",2.00,NaN,0.828652,0.867218,0.880507,0.873812,1.290106,NaN,NaN,NaN,hinge
108,LinearSVM Loss Check,"{'C': 0.25, 'loss': 'squared_hinge'}","LinearSVM Loss Check|{""C"": 0.25, ""loss"": ""squa...",0.25,NaN,0.826931,0.862928,0.883472,0.873079,0.933877,NaN,NaN,NaN,squared_hinge


### Training

Fit the Linear Support Vector Machine model using the engineered combined feature representation of the training data.

In [44]:
# reused seed used in data split script
svm = LinearSVC(max_iter=5000, C=1.5, random_state=42, loss='hinge')
svm.fit(train_features, train_labels)

,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'hinge'
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.5
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forthe dual coordinate descent (if ``dual=True``). When ``dual=False`` theunderlying implementation of :class:`LinearSVC` is not random and``random_state`` has no effect on the results.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_iter max_iter: int, default=1000The maximum number of iterations to be run.",5000
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed t

### Evaluation

Generate predictions on the validation set for model evaluation, then evaluate model performance using a classification report and confusion matrix.

In [45]:
svm_predictions = svm.predict(val_features)
print(classification_report(val_labels, svm_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(val_labels, svm_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.75      0.72      0.74     10616
         PII       0.87      0.88      0.88     21926

    accuracy                           0.83     32542
   macro avg       0.81      0.80      0.81     32542
weighted avg       0.83      0.83      0.83     32542



,Predicted Safe,Predicted PII
Actual Safe,7684,2932
Actual PII,2584,19342


# Final Test Evaluation

After comparing the models on the validation set, the final model configurations are evaluated on a held out test set. These results will provide an unbiased estimate of each model's performance on unseen data.

#### Baseline

In [46]:
baseline_test_predictions = np.ones(len(test_labels), dtype=int)
# zero_division set to 0 to hide warning due to zero examples being predicted as safe
print(classification_report(test_labels, baseline_test_predictions, zero_division=0))

cm = confusion_matrix(test_labels, baseline_test_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

           0       0.00      0.00      0.00     10617
           1       0.67      1.00      0.81     21926

    accuracy                           0.67     32543
   macro avg       0.34      0.50      0.40     32543
weighted avg       0.45      0.67      0.54     32543



,Predicted Safe,Predicted PII
Actual Safe,0,10617
Actual PII,0,21926


#### Logistic Regression

In [47]:
lr_test_predictions = lr.predict(test_features)
print(classification_report(test_labels, lr_test_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(test_labels, lr_test_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.76      0.73      0.74     10617
         PII       0.87      0.89      0.88     21926

    accuracy                           0.84     32543
   macro avg       0.81      0.81      0.81     32543
weighted avg       0.83      0.84      0.84     32543



,Predicted Safe,Predicted PII
Actual Safe,7783,2834
Actual PII,2502,19424


#### Naive Bayes

In [48]:
nb_test_predictions = nb.predict(test_features)
print(classification_report(test_labels, nb_test_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(test_labels, nb_test_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.63      0.60      0.61     10617
         PII       0.81      0.83      0.82     21926

    accuracy                           0.75     32543
   macro avg       0.72      0.71      0.72     32543
weighted avg       0.75      0.75      0.75     32543



,Predicted Safe,Predicted PII
Actual Safe,6318,4299
Actual PII,3742,18184


#### Linear SVM

In [49]:
svm_test_predictions = svm.predict(test_features)
print(classification_report(test_labels, svm_test_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(test_labels, svm_test_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.76      0.73      0.74     10617
         PII       0.87      0.89      0.88     21926

    accuracy                           0.83     32543
   macro avg       0.81      0.81      0.81     32543
weighted avg       0.83      0.83      0.83     32543



,Predicted Safe,Predicted PII
Actual Safe,7716,2901
Actual PII,2500,19426


## Results Summary Chart

In [50]:
predictions = {
    'Baseline': baseline_test_predictions,
    'Logistic Regression': lr_test_predictions,
    'Naive Bayes': nb_test_predictions,
    'Linear SVM': svm_test_predictions
}

comparison_results = []

for model, p in predictions.items():
    comparison_results.append({
        'Model': model,
        'Accuracy': accuracy_score(test_labels, p),
        'PII Precision': precision_score(test_labels, p, zero_division=0),
        'PII Recall': recall_score(test_labels, p, zero_division=0),
        'PII F1': f1_score(test_labels, p, zero_division=0)
    })

comparison_results = pd.DataFrame(comparison_results).round(4)

comparison_results

,Model,Accuracy,PII Precision,PII Recall,PII F1
0,Baseline,0.6738,0.6738,1.0000,0.8051
1,Logistic Regression,0.8360,0.8727,0.8859,0.8792
2,Naive Bayes,0.7529,0.8088,0.8293,0.8189
3,Linear SVM,0.8340,0.8701,0.8860,0.8780
